# Customer LTV — Two-Stage Model Training

End-to-end training pipeline:
1. Load config + data
2. Build feature matrix (RFM + behavioral + temporal + delivery)
3. Chronological train/test split
4. Stage 1 — Return classifier (binary XGBoost, PR-AUC primary metric)
5. Stage 2 — Spend tier classifier (3-class XGBoost, weighted F1)
6. Save model artifact
7. Log all params + metrics to MLflow

In [7]:
import sys
sys.path.append("..")

import os
import pandas as pd
import mlflow
import mlflow.xgboost

from src.data import load_config, compute_ltv_targets, chronological_split
from src.features import build_feature_matrix
from src.model import (
    prepare_data,
    select_features_shap,
    tune_hyperparameters,
    train_model,
    evaluate_stage1,
    evaluate_stage2,
    save_model,
)

cfg = load_config("../config.yaml")
print("Config loaded.")

Config loaded.


## 1. Load Data

In [8]:
abt    = pd.read_parquet("../data/processed/abt.parquet")
cutoff = pd.Timestamp(cfg["data"]["cutoff_date"])

print(f"ABT         : {abt.shape[0]:,} rows × {abt.shape[1]} cols")
print(f"Cutoff date : {cutoff.date()}")
print(f"Horizon     : {cfg['data']['horizon_days']} days")

ABT         : 96,478 rows × 24 cols
Cutoff date : 2018-07-01
Horizon     : 90 days


## 2. Compute Targets + Split

In [9]:
targets          = compute_ltv_targets(abt, cutoff_date=cutoff, horizon_days=cfg["data"]["horizon_days"])
train_t, test_t  = chronological_split(targets, abt, train_frac=cfg["split"]["train_frac"])

Eligible customers (have pre-cutoff orders): 81,265
Returners  (will_return=1): 280 (0.3%)
Non-returners (will_return=0): 80,985
Spend tiers (returners only): Low=94  Mid=92  High=94
Train customers : 65,012 | returners: 197 (0.3%)
Test  customers : 16,253  | returners: 83 (0.5%)


## 3. Build Feature Matrix

In [10]:
features     = build_feature_matrix(abt, cutoff)
feature_cols = [c for c in features.columns if c != "customer_unique_id"]

train = train_t.merge(features, on="customer_unique_id", how="left")
test  = test_t.merge(features,  on="customer_unique_id", how="left")

print(f"Train : {len(train):,} customers")
print(f"Test  : {len(test):,} customers")

Feature matrix: 81,265 customers × 22 features
Train : 65,012 customers
Test  : 16,253 customers


## 4. Stage 1 — Return Classifier

In [11]:
# Prepare
X1_train, y1_train, _ = prepare_data(train[feature_cols], train["will_return"])
X1_test = test[X1_train.columns]

# SHAP feature selection — top 15 of 29
# Tested: top_n=20 and top_n=21 both hurt PR-AUC with only 197 positives
# SHAP ranks all 29 features; new features displace weaker ones if they're more informative
os.makedirs("../models", exist_ok=True)
feats1     = select_features_shap(X1_train, y1_train, top_n=15, cache_path="../models/shap_stage1.json")
X1_train_s = X1_train[feats1]
X1_test_s  = X1_test[feats1]

print(f"Stage 1 features ({len(feats1)}): {feats1}")

SHAP features loaded from cache (15 features)
Stage 1 features (15): ['avg_delivery_delay_days', 'recency', 'avg_delivery_speed_days', 'avg_freight_ratio', 'monetary', 'avg_review_score', 'avg_order_value', 'pct_installments', 'avg_order_gap_days', 'customer_state_freq', 'has_voucher', 'unique_categories', 'pct_late_deliveries', 'avg_items_per_order', 'spend_last_60d']


In [12]:
n_trials = cfg["tuning"]["n_trials"]
cv_folds = cfg["tuning"]["cv_folds"]

params1 = tune_hyperparameters(X1_train_s, y1_train, objective="binary", n_trials=n_trials, cv_folds=cv_folds)
model1  = train_model(X1_train_s, y1_train, params1, objective="binary")
m1      = evaluate_stage1(model1, X1_test_s, test["will_return"])

print(f"\nStage 1 Results")
print(f"  PR-AUC    : {m1['pr_auc']}")
print(f"  ROC-AUC   : {m1['roc_auc']}")
print(f"  Precision : {m1['precision']}")
print(f"  Recall    : {m1['recall']}")
print(f"  Threshold : {m1['threshold']}")

Best score (binary): 0.0133

Stage 1 Results
  PR-AUC    : 0.0057
  ROC-AUC   : 0.5188
  Precision : 0.0057
  Recall    : 0.6747
  Threshold : 0.0875


## 5. Stage 2 — Spend Tier Classifier

In [13]:
# Returners only
train_r = train[train["will_return"] == 1].reset_index(drop=True)
test_r  = test[test["will_return"]  == 1].reset_index(drop=True)
print(f"Returners — train: {len(train_r)}, test: {len(test_r)}")

# Prepare
X2_train, y2_train, _ = prepare_data(train_r[feature_cols], train_r["spend_tier"])
X2_test = test_r[X2_train.columns]

# SHAP feature selection — top 10 of 29
# Tested: top_n=15 and top_n=20 both hurt F1 on 197 training samples
feats2     = select_features_shap(X2_train, y2_train, top_n=10, cache_path="../models/shap_stage2.json")
X2_train_s = X2_train[feats2]
X2_test_s  = test_r[feats2]

print(f"Stage 2 features ({len(feats2)}): {feats2}")

Returners — train: 197, test: 83
Top 10 SHAP features selected
Stage 2 features (10): ['avg_delivery_delay_days', 'avg_freight_ratio', 'monetary', 'recency', 'avg_order_value', 'avg_delivery_speed_days', 'customer_state_freq', 'avg_review_score', 'pct_installments', 'avg_items_per_order']


In [14]:
params2 = tune_hyperparameters(X2_train_s, y2_train, objective="multi", n_trials=n_trials, cv_folds=3)
model2  = train_model(X2_train_s, y2_train, params2, objective="multi")
m2      = evaluate_stage2(model2, X2_test_s, test_r["spend_tier"])

print(f"\nStage 2 Results")
print(f"  Weighted F1 : {m2['weighted_f1']}")
print(f"  F1 Low      : {m2['f1_low']}")
print(f"  F1 Mid      : {m2['f1_mid']}")
print(f"  F1 High     : {m2['f1_high']}")

Best score (multi): 0.4650

Stage 2 Results
  Weighted F1 : 0.447
  F1 Low      : 0.5
  F1 Mid      : 0.2424
  F1 High     : 0.5507


## 6. Save Model

In [15]:
save_model(
    model1, model2,
    feats1, feats2,
    threshold=m1["threshold"],
    path="../models/ltv_model.pkl",
)

Model saved → ../models/ltv_model.pkl


## 7. MLflow Logging

In [16]:
mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("customer-ltv")

with mlflow.start_run(run_name="two-stage-xgboost"):

    # Pipeline parameters
    mlflow.log_params({
        "cutoff_date":   cfg["data"]["cutoff_date"],
        "horizon_days":  cfg["data"]["horizon_days"],
        "train_frac":    cfg["split"]["train_frac"],
        "n_trials":      n_trials,
        "cv_folds":      cv_folds,
        "stage1_top_n":  len(feats1),
        "stage2_top_n":  len(feats2),
    })

    # Stage 1 best hyperparams
    mlflow.log_params({f"s1_{k}": v for k, v in params1.items()})

    # Stage 2 best hyperparams
    mlflow.log_params({f"s2_{k}": v for k, v in params2.items()})

    # Stage 1 metrics
    mlflow.log_metrics({
        "s1_pr_auc":    m1["pr_auc"],
        "s1_roc_auc":   m1["roc_auc"],
        "s1_precision": m1["precision"],
        "s1_recall":    m1["recall"],
        "s1_threshold": m1["threshold"],
    })

    # Stage 2 metrics
    mlflow.log_metrics({
        "s2_weighted_f1": m2["weighted_f1"],
        "s2_f1_low":      m2["f1_low"],
        "s2_f1_mid":      m2["f1_mid"],
        "s2_f1_high":     m2["f1_high"],
    })

    # Artifacts
    mlflow.log_artifact("../models/ltv_model.pkl")
    mlflow.log_artifact("../models/shap_stage1.json")
    mlflow.log_artifact("../models/shap_stage2.json")

    run_id = mlflow.active_run().info.run_id
    print(f"MLflow run logged: {run_id}")
    print(f"\nSummary")
    print(f"  Stage 1  PR-AUC      : {m1['pr_auc']}")
    print(f"  Stage 1  ROC-AUC     : {m1['roc_auc']}")
    print(f"  Stage 2  Weighted F1 : {m2['weighted_f1']}")

MLflow run logged: b8e7a253ff9249629a708d25e046e2a1

Summary
  Stage 1  PR-AUC      : 0.0057
  Stage 1  ROC-AUC     : 0.5188
  Stage 2  Weighted F1 : 0.447
